In [1]:
import os
import json
import pandas as pd
import glob

cwd = os.getcwd()
data_dir = f"{cwd}/data"

meta_df = pd.read_excel(data_dir+"/ASPECT_PolicieswIDs_3Mar26.xlsx")

We're gonna start with just one json but later we'll loop all the jsons in on this action

In [35]:
with open(f"{data_dir}/output_jsons/254.json", encoding="utf-8") as f:
    mineru_pol = json.load(f)

In [36]:
# each policy item is a list of page items
# each page item is a dictionary containing the keys "page" and "content"
# the "content" for each page is a list of content items
# each content item is a dictionary containing the keys "type", "angle", and "content"
# the "type" field options can be found here: https://github.com/opendatalab/mineru-vl-utils/blob/main/mineru_vl_utils/structs.py
sents = []
partial = False
doc_id = "#"
for page_itm in mineru_pol:
    pg = str(page_itm['page'])
    for cont_itm in page_itm['content']:
        # reset partial to False after titles so that text before titles [which usually don't end with periods] 
        # don't merge with our first body text sentence
        if cont_itm['type']=="title":
            partial = False
        if cont_itm['type']=="text":
            text = cont_itm['content']
            if partial:
                sents[-1]['text'] += " "+text
                if sents[-1]['pg'][-1] != pg:
                    sents[-1]['pg'] += f"-{pg}"
            else:
                sents.append({"text": text, "pg": pg, "doc_id": doc_id})
            # if the text is incomplete (i.e. we will need to include the next content item)
            if text[-1] != ".":
                partial = True
            else:
                partial = False
sents

[{'text': 'United Nations Environment Assembly of the United Nations Environment Programme Fourth session Nairobi, 11-15 March 2019',
  'pg': '1',
  'doc_id': '#'},
 {'text': 'The United Nations Environment Assembly, Recalling the commitment made by Heads of State and Government in the outcome document of the United Nations Conference on Sustainable Development, entitled "The future we want", which recognized the importance of facilitating ecosystem conservation, regeneration, and restoration and resilience in the face of new and emerging challenges, Recognizing that peatlands exist in more than 180 countries in different regions of the world, and that, while peatlands cover only some 3 per cent of global land area,[2] they contain a far higher proportion of global organic soil carbon, making them one of the world\'s largest carbon stores, contributing to global climate change mitigation through carbon sequestration.',
  'pg': '1',
  'doc_id': '#'},
 {'text': 'Recognizing also that deg

Let's loop

In [ ]:
peat_doc_sents = []
for pol_id in [254, 316, 363, 1071, 1182]:
    with open(f"{data_dir}/output_jsons/{pol_id}.json", "r", encoding="utf-8") as f:
        mineru_pol = json.load(f)
    partial = False
    for page_itm in mineru_pol:
        pg = str(page_itm['page'])
        for cont_itm in page_itm['content']:
            # reset partial to False after titles so that text before titles [which usually don't end with periods] 
            # don't merge with our first body text sentence
            if cont_itm['type']=="title":
                partial = False
            if cont_itm['type']=="text":
                text = cont_itm['content']
                if partial:
                    peat_doc_sents[-1]['text'] += " "+text
                    if peat_doc_sents[-1]['pg'][-1] != pg:
                        peat_doc_sents[-1]['pg'] += f"-{pg}"
                else:
                    peat_doc_sents.append({"text": text, "pg": pg, "doc_id": pol_id, "label":[]})
                # if the text is incomplete (i.e. we will need to include the next content item)
                if text[-1] != ".":
                    partial = True
                else:
                    partial = False
len(peat_doc_sents)

In [49]:
with open(f"{data_dir}/peat_doc_sents.jsonl", "w", encoding="utf-8") as f:
    for sample in peat_doc_sents:
        json.dump(sample, f)
        f.write("\n")

In [8]:
with open(f"{data_dir}/peat_doc_sents.jsonl", "r", encoding="utf-8") as f:
    pged_data = [json.loads(line) for line in f]

doccano_data = [{"text":entry["text"], "label":entry['label']} for entry in pged_data]

with open(f"{data_dir}/peat_sents.jsonl", "w", encoding="utf-8") as f:
    for sample in doccano_data:
        json.dump(sample, f)
        f.write("\n")